In [0]:
use catalog classic_workspace;
use schema default;

In [0]:
DROP TABLE IF EXISTS customers;

CREATE OR REPLACE TABLE customers (
  customer_id  INT     COMMENT 'Primary key',
  name         STRING  COMMENT 'Full name',
  email        STRING  COMMENT 'Contact email',
  tier         STRING  COMMENT 'bronze / silver / gold',
  updated_at   TIMESTAMP COMMENT 'Last modified timestamp'
)
USING DELTA
COMMENT 'Customer master — target for MERGE demos';

INSERT INTO customers VALUES
  (1, 'Alice Nguyen',  'alice@example.com',  'silver', '2024-06-01 08:00:00'),
  (2, 'Bob Patel',     'bob@example.com',    'bronze', '2024-06-01 08:00:00'),
  (3, 'Carol Santos',  'carol@example.com',  'gold',   '2024-06-01 08:00:00'),
  (4, 'David Kim',     'david@example.com',  'bronze', '2024-06-01 08:00:00'),
  (5, 'Eva Müller',    'eva@example.com',    'silver', '2024-06-01 08:00:00');

In [0]:
select * from customers;

In [0]:
%python
from pyspark.sql import Row
from datetime import datetime

updates = [
    # customer_id 2: email changed, upgrade to silver
    Row(customer_id=2, name='Bob Patel',
        email='bob.patel@newdomain.com', tier='silver',
        updated_at=datetime(2024, 6, 15, 9, 0, 0)),
    # customer_id 3: tier upgraded to platinum
    Row(customer_id=3, name='Carol Santos',
        email='carol@example.com', tier='platinum',
        updated_at=datetime(2024, 6, 15, 9, 0, 0)),
    # customer_id 6: brand new customer
    Row(customer_id=6, name='Frank Osei',
        email='frank@example.com', tier='bronze',
        updated_at=datetime(2024, 6, 15, 9, 0, 0)),
]

df_updates = spark.createDataFrame(updates)
df_updates.createOrReplaceTempView("customer_updates")
print("Source batch ready — 2 updates, 1 new record.")

In [0]:
select * from customer_updates;

In [0]:
MERGE INTO customers AS target          -- the table being modified
USING customer_updates AS source        -- the incoming batch
ON target.customer_id = source.customer_id  -- the join key

WHEN MATCHED THEN                       -- source row found a match in target
  UPDATE SET
    target.name       = source.name,
    target.email      = source.email,
    target.tier       = source.tier,
    target.updated_at = source.updated_at

WHEN NOT MATCHED THEN                   -- source row has no match in target
  INSERT (customer_id, name, email, tier, updated_at)
  VALUES (source.customer_id, source.name, source.email,
          source.tier, source.updated_at);

In [0]:
SELECT * FROM customers ORDER BY customer_id;

In [0]:
CREATE OR REPLACE TABLE customers (
  customer_id  INT,
  name         STRING,
  email        STRING,
  tier         STRING,
  updated_at   TIMESTAMP
)
USING DELTA;

INSERT INTO customers VALUES
  (1, 'Alice Nguyen',  'alice@example.com',  'silver',   '2024-06-15 09:00:00'),
  (2, 'Bob Patel',     'bob@example.com',    'silver',   '2024-06-15 09:00:00'),
  (3, 'Carol Santos',  'carol@example.com',  'platinum', '2024-06-15 09:00:00'),
  (4, 'David Kim',     'david@example.com',  'bronze',   '2024-06-15 09:00:00'),
  (5, 'Eva Müller',    'eva@example.com',    'silver',   '2024-06-15 09:00:00');

In [0]:
%python
from pyspark.sql import Row
from datetime import datetime

late_batch = [
    # ID 1: fresh update — should be applied
    Row(customer_id=1, name='Alice Nguyen',
        email='alice.new@example.com', tier='gold',
        updated_at=datetime(2024, 6, 20, 10, 0, 0)),
    # ID 2: stale — source timestamp is OLDER than target (late arrival)
    # target has 2024-06-15, source has 2024-05-28 — should be SKIPPED
    Row(customer_id=2, name='Bob Patel',
        email='bob.stale@example.com', tier='bronze',
        updated_at=datetime(2024, 5, 28, 6, 0, 0)),
    # ID 6: new customer — should be inserted
    Row(customer_id=6, name='Grace Lin',
        email='grace@example.com', tier='bronze',
        updated_at=datetime(2024, 6, 20, 10, 0, 0)),
]

df_late = spark.createDataFrame(late_batch)
df_late.createOrReplaceTempView("crm_sync")
print("CRM sync batch ready.")

In [0]:
select * from crm_sync

In [0]:
MERGE INTO customers AS target
USING crm_sync AS source
ON target.customer_id = source.customer_id

WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    target.name       = source.name,
    target.email      = source.email,
    target.tier       = source.tier,
    target.updated_at = source.updated_at

WHEN NOT MATCHED THEN
  INSERT (customer_id, name, email, tier, updated_at)
  VALUES (source.customer_id, source.name, source.email,
          source.tier, source.updated_at);

In [0]:
SELECT * FROM customers ORDER BY customer_id;

In [0]:
%python
from pyspark.sql import Row
from datetime import datetime
full_sync = [
    Row(customer_id=1, name='Alice Nguyen',
        email='alice.new@example.com', tier='gold',
        updated_at=datetime(2024, 6, 21, 8, 0, 0)),
    Row(customer_id=2, name='Bob Patel',
        email='bob@example.com', tier='silver',
        updated_at=datetime(2024, 6, 21, 8, 0, 0)),
    Row(customer_id=3, name='Carol Santos',
        email='carol@example.com', tier='platinum',
        updated_at=datetime(2024, 6, 21, 8, 0, 0)),
    Row(customer_id=4, name='David Kim',
        email='david@example.com', tier='bronze',
        updated_at=datetime(2024, 6, 21, 8, 0, 0)),
    Row(customer_id=5, name='Eva Müller',
        email='eva@example.com', tier='silver',
        updated_at=datetime(2024, 6, 21, 8, 0, 0)),
    # customer_id 6 (Grace) is deliberately absent — deleted upstream
]

df_full = spark.createDataFrame(full_sync)
df_full.createOrReplaceTempView("full_sync_source")

# Show current target count before MERGE
current_count = spark.table("customers").count()
print(f"Target row count before MERGE: {current_count}")

In [0]:
SELECT * FROM customers;

In [0]:
MERGE INTO customers AS target
USING full_sync_source AS source
ON target.customer_id = source.customer_id

WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    target.name       = source.name,
    target.email      = source.email,
    target.tier       = source.tier,
    target.updated_at = source.updated_at

WHEN NOT MATCHED BY TARGET THEN
  INSERT (customer_id, name, email, tier, updated_at)
  VALUES (source.customer_id, source.name, source.email,
          source.tier, source.updated_at)

WHEN NOT MATCHED BY SOURCE THEN
  DELETE;

In [0]:
SELECT * FROM customers ORDER BY customer_id;